# Managing Conversation State with OpenAI's API

Two approaches for managing conversation state:

1. **Manually managing conversation state:** Building a conversation history list and appending messages for each turn.
2. **Chaining responses using `previous_response_id`:** This method allows the model to access prior context automatically.

## 1. Manual Conversation State Management

For this first example, we'll create a simple knock-knock joke conversation by manually constructing a conversation history. Every turn in the conversation is stored in a list of messages, where each message is a dictionary containing a role (`user` or `assistant`) and the content.

This way of managing state is very explicit and works well when you want fine-grained control over the conversation history.

In [ ]:
%pip install -q openai

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

# API Key Setup - uses environment variable or prompts for input
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

# Client and Model Configuration
client = OpenAI()
MODEL = "gpt-5.2"

In [4]:
# Construct a conversation history manually for a knock-knock joke
history = [
    {"role": "user", "content": "knock knock."},
    {"role": "assistant", "content": "Who's there?"},
    {"role": "user", "content": "Orange."}
]

# Create a response using the conversation history
response = client.responses.create(
    model=MODEL,
    input=history
)

print("Assistant response:", response.output_text)

# (Optional) Inspect the full response object for further details
# print(response)

Assistant response: Orange who?


## 2. Updating Conversation History Across Rounds

In this example, we'll start a conversation with a simple joke prompt. We'll then update the conversation history with the model's response, and ask for another joke. Updating the conversation history manually makes sure the model has full context of previous turns.

Let's see how this works.

### Understanding the `store` Parameter

In the code below, you'll see `store=False`. This parameter controls whether OpenAI saves your conversation:

- **`store=False`**: OpenAI will **not** save this conversation. The request is processed and discarded. Use this for sensitive data or when you don't need conversation history on OpenAI's servers.

- **`store=True`** (default): OpenAI saves the conversation, making it accessible later via the API. This enables features like conversation retrieval and is useful for building persistent chat applications.

Choose based on your privacy requirements and whether you need to access the conversation history through OpenAI's dashboard or API.

In [6]:
# Start a new conversation with an initial joke request
history = [
    {"role": "user", "content": "tell me a joke"}
]

response = client.responses.create(
    model=MODEL,
    input=history,
    store=False  # We're not storing the conversation automatically
)

print("Joke:", response.output_text)

# Update the conversation history with all parts of the response
history += [{"role": el.role, "content": el.content} for el in response.output]

# Ask for another joke in the same conversation
history.append({"role": "user", "content": "tell me another"})

second_response = client.responses.create(
    model=MODEL,
    input=history,
    store=False
)

print("Another joke:", second_response.output_text)

Joke: Why did the scarecrow win an award?  

Because he was outstanding in his field!


Another joke: Why don’t scientists trust atoms?  

Because they make up everything!


## 3. Chaining Responses with `previous_response_id`

Another way to carry conversation context is to chain responses by using the `previous_response_id` parameter. With this method, you initiate a conversation and then provide additional requests that reference the previous response.

Below is an example where we first ask the model for a joke and then ask it to explain why that joke is funny.

In [7]:
# Ask for a joke
response = client.responses.create(
    model=MODEL,
    input="tell me a joke"
)

print("Joke:", response.output_text)

# Chain the next request using previous_response_id so the model retains context
second_response = client.responses.create(
    model=MODEL,
    previous_response_id=response.id,
    input=[{"role": "user", "content": "explain why this is funny."}]
)

print("Explanation:", second_response.output_text)

Joke: Why did the scarecrow win an award? 

Because he was outstanding in his field!


Explanation: The joke plays on a clever double meaning. The phrase "outstanding in his field" can mean that the scarecrow is excellent at his job of scaring away birds (hence winning an award), but it can also be taken literally to mean that he is literally standing out in a field. The unexpected twist of interpreting "outstanding" both ways creates humor through wordplay.


## Final Notes

- **Managing Tokens:** Keep in mind that every input and output token (and any internal reasoning tokens) counts toward your model's context window. For long conversations, you may need to manage or truncate history.
- **Best Practices:** When building production applications, consider automating state management so that conversations flow naturally. Using proper error handling, logging, and context management will improve your application's DX significantly.

This notebook demonstrated how you can use Python to manage conversation state. Feel free to experiment with other conversation flows and adjust the approach as needed for your teaching or production environments.